In [8]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from src.changepoint import detect_changepoints
from src.discovery import (
    DiscoveryAlgorithm,
    discover_model,
    save_model_visualization,
)
from src.merging import (
    SegmentProfile,
    build_compatibility_graph,
    compute_profiles,
    minimum_clique_cover,
)
from src.smt import get_solver
from src.synthesis import IntervalRule, synthesize_rules, evaluate_rule
from src.trace import build_traces

In [9]:
# ── CASAS file(s) ────────────────────────────────────────────────────
CASAS_DIR = Path("data/CASAS")
# Use one or more files from the same household / sensor set.
# All files listed here must share the same sensor names.
CASAS_FILES = ["hh101.csv"]

# ── Resampling ───────────────────────────────────────────────────────
RESAMPLE_INTERVAL = "1min"   # pandas offset alias (e.g. "30s", "1min", "5min")

# ── Day / trace selection ────────────────────────────────────────────
MAX_DAYS: int | None = 14   # set to None for the full dataset

# ── Sensor / sample control ─────────────────────────────────────────
MAX_SENSORS: int | None = None   # limit number of sensors (None = all)
DOWNSAMPLE_FACTOR: int = 2       # take every N-th sample for PELT (1 = no downsampling)

# ── Pipeline parameters ─────────────────────────────────────────────
PENALTY = 5.0               # PELT penalty (lower → more changepoints)
MIN_SEGMENT_SIZE = 5        # minimum samples between changepoints
PELT_MODEL = "l2"           # cost model: "l2" (fast, O(N)) or "rbf" (slow, O(N²))
SMT_TIMEOUT_MS = 10_000     # 10 s per SMT query
ALGORITHM = DiscoveryAlgorithm.INDUCTIVE
OUTPUT_PATH = "casas_model.png"

In [10]:
# ── Value encoding ────────────────────────────────────────────────────
VALUE_MAP = {"ON": 1.0, "OPEN": 1.0, "OFF": 0.0, "CLOSE": 0.0}


def _encode_value(v: str) -> float:
    """Map a CASAS sensor value to a float.

    Categorical values (ON/OFF/OPEN/CLOSE) are mapped via VALUE_MAP.
    Numeric strings (e.g. temperature readings) are parsed directly.
    """
    v = v.strip()
    if v in VALUE_MAP:
        return VALUE_MAP[v]
    try:
        return float(v)
    except ValueError:
        return np.nan


# ── Data loading ─────────────────────────────────────────────────────

def load_casas_file(path: Path) -> pd.DataFrame:
    """Parse a single CASAS CSV into a long-format DataFrame.

    Auto-detects whether the file is comma- or space-delimited.

    Returns
    -------
    pd.DataFrame
        Columns: ``timestamp`` (DatetimeIndex), ``sensor``, ``value``.
    """
    with open(path, "r") as f:
        first_line = f.readline()

    if "," in first_line:
        # Comma-delimited: date,time,sensor,value
        df = pd.read_csv(
            path,
            header=None,
            names=["date", "time", "sensor", "raw_value"],
        )
        df["timestamp"] = pd.to_datetime(df["date"] + " " + df["time"], format="mixed")
    else:
        # Space-delimited: date time sensor value
        df = pd.read_csv(
            path,
            header=None,
            sep=r"\s+",
            names=["date", "time", "sensor", "raw_value"],
        )
        df["timestamp"] = pd.to_datetime(df["date"] + " " + df["time"], format="mixed")

    df["value"] = df["raw_value"].astype(str).map(_encode_value)
    return df[["timestamp", "sensor", "value"]].dropna(subset=["value"])


def load_casas_files(directory: Path, filenames: list[str]) -> pd.DataFrame:
    """Load and concatenate multiple CASAS files."""
    frames = []
    for fname in filenames:
        df = load_casas_file(directory / fname)
        print(f"  {fname}: {len(df):,} events, "
              f"{df['sensor'].nunique()} sensors, "
              f"{df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
        frames.append(df)
    return pd.concat(frames, ignore_index=True).sort_values("timestamp")


# ── Event → time-series conversion ──────────────────────────────────

def events_to_timeseries(
    events: pd.DataFrame,
    resample_interval: str,
) -> pd.DataFrame:
    """Convert long-format events into a resampled wide-format time series.

    Each sensor becomes a column.  The most recent value is forward-filled
    across the regular time grid.
    """
    # Pivot: one column per sensor, timestamp as index
    wide = events.pivot_table(
        index="timestamp",
        columns="sensor",
        values="value",
        aggfunc="last",
    )
    # Forward-fill sparse events, then resample to a regular grid
    wide = wide.ffill()
    wide = wide.resample(resample_interval).last().ffill().bfill()
    return wide


# ── Day splitting ────────────────────────────────────────────────────

def split_by_day(
    ts: pd.DataFrame,
    max_days: int | None = None,
) -> list[np.ndarray]:
    """Split a time-series DataFrame into per-day numpy arrays.

    Parameters
    ----------
    ts : pd.DataFrame
        Wide-format time series with a DatetimeIndex.
    max_days : int | None
        Limit the number of days returned.

    Returns
    -------
    list[np.ndarray]
        One array of shape ``(N_minutes, N_sensors)`` per day.
    """
    ts = ts.copy()
    ts["_date"] = ts.index.date
    grouped = ts.groupby("_date")
    days: list[np.ndarray] = []
    for _, group in grouped:
        arr = group.drop(columns="_date").to_numpy(dtype=np.float64)
        if arr.shape[0] >= 2:  # skip degenerate days
            days.append(arr)
    if max_days is not None:
        days = days[:max_days]
    return days


# ── Normalisation ────────────────────────────────────────────────────

@dataclass
class Normaliser:
    lo: np.ndarray
    span: np.ndarray

    @classmethod
    def fit(cls, data: np.ndarray) -> "Normaliser":
        lo = data.min(axis=0)
        hi = data.max(axis=0)
        span = hi - lo
        span[span == 0] = 1.0
        return cls(lo=lo, span=span)

    def transform(self, data: np.ndarray) -> np.ndarray:
        return (data - self.lo) / self.span

    def label(self, col: int, val: float) -> float:
        return val * self.span[col] + self.lo[col]


# ── Rule helpers (same as Future Factory notebook) ───────────────────

def bounding_box_rules(
    classes: list[list[int]],
    profiles: list[SegmentProfile],
    n_vars: int,
) -> list[IntervalRule]:
    """Bounding-box rule per class (fallback when SMT returns UNSAT)."""
    rules: list[IntervalRule] = []
    for i, members in enumerate(classes):
        class_profiles = [profiles[idx] for idx in members]
        lo = np.min([p.lo for p in class_profiles], axis=0).tolist()
        hi = np.max([p.hi for p in class_profiles], axis=0).tolist()
        rules.append(IntervalRule(lo=lo, hi=hi, class_id=i))
    return rules


def try_synthesize_rules(
    classes: list[list[int]],
    profiles: list[SegmentProfile],
    n_vars: int,
    timeout_ms: int | None,
) -> tuple[list[IntervalRule], bool]:
    """Try exact SMT synthesis; fall back to bounding boxes on UNSAT."""
    smt = get_solver()
    try:
        rules = synthesize_rules(smt, classes, profiles, n_vars, timeout_ms)
        return rules, True
    except RuntimeError:
        rules = bounding_box_rules(classes, profiles, n_vars)
        return rules, False


def build_traces_nearest(
    data: np.ndarray,
    rules: list[IntervalRule],
) -> list[list[int]]:
    """Build traces with nearest-center fallback for ambiguous points."""
    centers = np.array([
        [(r.lo[k] + r.hi[k]) / 2 for k in range(len(r.lo))]
        for r in rules
    ])
    n = data.shape[0]
    trace: list[int] = []
    prev = -1
    for t in range(n):
        pt = data[t]
        matching = [r for r in rules if evaluate_rule(r, pt)]
        if len(matching) == 1:
            active = matching[0].class_id
        else:
            dists = np.linalg.norm(centers - pt, axis=1)
            active = rules[int(np.argmin(dists))].class_id
        if active != prev:
            trace.append(active)
            prev = active
    return [trace]


def per_day_traces(
    days: list[np.ndarray],
    rules: list[IntervalRule],
) -> list[list[int]]:
    """Build one trace per day using the shared rules."""
    all_traces: list[list[int]] = []
    for day_data in days:
        all_traces.extend(build_traces_nearest(day_data, rules))
    return all_traces


def format_rule(rule: IntervalRule, norm: Normaliser, names: list[str]) -> str:
    parts: list[str] = []
    for k in range(len(rule.lo)):
        lo_orig = norm.label(k, rule.lo[k])
        hi_orig = norm.label(k, rule.hi[k])
        parts.append(f"{names[k]}∈[{lo_orig:.2f},{hi_orig:.2f}]")
    return " ∧ ".join(parts)

In [11]:
# ── 1. Load CASAS data ───────────────────────────────────────────────
print(f"Loading CASAS files from {CASAS_DIR} ...")
events = load_casas_files(CASAS_DIR, CASAS_FILES)
sensor_names = sorted(events["sensor"].unique().tolist())

# ── Apply sensor limit ───────────────────────────────────────────────
if MAX_SENSORS is not None and len(sensor_names) > MAX_SENSORS:
    sensor_names = sensor_names[:MAX_SENSORS]
    events = events[events["sensor"].isin(sensor_names)]
    print(f"  (limited to {MAX_SENSORS} sensors)")

print(f"\nSensors ({len(sensor_names)}): {sensor_names}")
print(f"Total events: {len(events):,}")

# ── 2. Resample to time series ──────────────────────────────────────
print(f"\nResampling to {RESAMPLE_INTERVAL} intervals ...")
ts = events_to_timeseries(events, RESAMPLE_INTERVAL)
# Ensure consistent column order
ts = ts[sensor_names]
print(f"Time-series shape: {ts.shape[0]:,} rows × {ts.shape[1]} sensors")
print(f"Date range: {ts.index.min()} → {ts.index.max()}")

# ── 3. Split by day & normalise ─────────────────────────────────────
days_raw = split_by_day(ts, max_days=MAX_DAYS)
print(f"\n{len(days_raw)} day(s) selected")
for i, d in enumerate(days_raw):
    print(f"  Day {i}: {d.shape[0]} samples × {d.shape[1]} sensors")

concat_raw = np.vstack(days_raw)
norm = Normaliser.fit(concat_raw)
days = [norm.transform(d) for d in days_raw]
concat = np.vstack(days)
n_sensors = concat.shape[1]
print(f"\nNormalised concatenation: {concat.shape[0]:,} samples × "
      f"{n_sensors} sensors")

Loading CASAS files from data/CASAS ...
  hh101.csv: 1,286,244 events, 6 sensors, 2012-07-18 → 2013-07-25

Sensors (6): ['Bathroom', 'Bedroom', 'DiningRoom', 'Kitchen', 'LivingRoom', 'OutsideDoor']
Total events: 1,286,244

Resampling to 1min intervals ...
Time-series shape: 535,622 rows × 6 sensors
Date range: 2012-07-18 12:54:00 → 2013-07-25 11:55:00

14 day(s) selected
  Day 0: 666 samples × 6 sensors
  Day 1: 1440 samples × 6 sensors
  Day 2: 1440 samples × 6 sensors
  Day 3: 1440 samples × 6 sensors
  Day 4: 1440 samples × 6 sensors
  Day 5: 1440 samples × 6 sensors
  Day 6: 1440 samples × 6 sensors
  Day 7: 1440 samples × 6 sensors
  Day 8: 1440 samples × 6 sensors
  Day 9: 1440 samples × 6 sensors
  Day 10: 1440 samples × 6 sensors
  Day 11: 1440 samples × 6 sensors
  Day 12: 1440 samples × 6 sensors
  Day 13: 1440 samples × 6 sensors

Normalised concatenation: 19,386 samples × 6 sensors


In [12]:
# ── 4. Phase 1 — change-point detection ─────────────────────────────
print(f"Phase 1: PELT (model={PELT_MODEL!r}, penalty={PENALTY}, "
      f"min_size={MIN_SEGMENT_SIZE}, downsample={DOWNSAMPLE_FACTOR})")
boundaries = detect_changepoints(
    concat, PENALTY, MIN_SEGMENT_SIZE,
    model=PELT_MODEL, downsample=DOWNSAMPLE_FACTOR,
)
n_segments = len(boundaries) - 1
print(f"  → {n_segments} segments from {len(boundaries)} boundaries")

# ── 5. Phase 2 — segment merging ────────────────────────────────────
print("\nPhase 2: Segment profiling + clique cover ...")
profiles = compute_profiles(concat, boundaries)
adj = build_compatibility_graph(profiles)
classes = minimum_clique_cover(adj, len(profiles))
n_classes = len(classes)
print(f"  → {n_segments} segments merged into {n_classes} equivalence classes")

Phase 1: PELT (model='l2', penalty=5.0, min_size=5, downsample=2)
  → 1 segments from 2 boundaries

Phase 2: Segment profiling + clique cover ...
  → 1 segments merged into 1 equivalence classes


In [13]:
# ── 6. Phase 3 — rule synthesis ──────────────────────────────────────
print(f"Phase 3: Rule synthesis ({n_classes} classes) ...")
rules, exact = try_synthesize_rules(classes, profiles, n_sensors, SMT_TIMEOUT_MS)
if exact:
    print("  (exact SMT discrimination)")
else:
    print("  (bounding-box fallback — exact discrimination UNSAT)")
print()
for rule in rules:
    print(f"  Rule {rule.class_id}: {format_rule(rule, norm, sensor_names)}")

Phase 3: Rule synthesis (1 classes) ...
  (exact SMT discrimination)

  Rule 0: Bathroom∈[0.00,1.00] ∧ Bedroom∈[0.00,1.00] ∧ DiningRoom∈[0.00,1.00] ∧ Kitchen∈[0.00,1.00] ∧ LivingRoom∈[0.00,1.00] ∧ OutsideDoor∈[0.00,1.00]


In [14]:
# ── 7. Per-day trace construction ────────────────────────────────────
print(f"Building per-day traces ({len(days)} days) ...")
traces = per_day_traces(days, rules)
print(f"  → {len(traces)} trace(s)")
for i, trace in enumerate(traces):
    print(f"  Day {i}: {len(trace)} events → {trace}")

# ── 8. Process discovery ─────────────────────────────────────────────
print(f"\nProcess discovery ({ALGORITHM.name}) ...")
net, im, fm = discover_model(traces, rules, ALGORITHM, sensor_names)
print(f"  → Petri net: {len(net.transitions)} transitions, "
      f"{len(net.places)} places")

# ── 9. Visualise ─────────────────────────────────────────────────────
save_model_visualization(net, im, fm, OUTPUT_PATH)
print(f"\nModel saved to {OUTPUT_PATH}")

Building per-day traces (14 days) ...
  → 14 trace(s)
  Day 0: 1 events → [0]
  Day 1: 1 events → [0]
  Day 2: 1 events → [0]
  Day 3: 1 events → [0]
  Day 4: 1 events → [0]
  Day 5: 1 events → [0]
  Day 6: 1 events → [0]
  Day 7: 1 events → [0]
  Day 8: 1 events → [0]
  Day 9: 1 events → [0]
  Day 10: 1 events → [0]
  Day 11: 1 events → [0]
  Day 12: 1 events → [0]
  Day 13: 1 events → [0]

Process discovery (INDUCTIVE) ...
  → Petri net: 1 transitions, 2 places

Model saved to casas_model.png
